# Elevation against external truth

This notebook holds the validation record — including the arc that
nearly convinced us the method was broken.

**You are here: 04.** The elevations themselves against external truth — including the validation artefact that nearly fooled us.

```text
+- the evidence chain ------------------------------------------------+
|  datacube -> water masks -> per-pixel wet/dry series                |
|    01 what is estimable  ->  02 boundary audit  ->  03 operator vs  |
|    gauges  ->  04 elevations vs truth  ->  05 uncertainty and       |
|    hydraulic layers  ->  06 negatives kept  ->  07 coast census     |
+---------------------------------------------------------------------+
```

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# run from the repo root so the results/ paths resolve
here = Path.cwd()
while not (here / "pyintertidal").is_dir():
    if here.parent == here:
        raise FileNotFoundError("repo root not found above " + str(Path.cwd()))
    here = here.parent
os.chdir(here)

def load(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

## 0. The compression that started it all
The first serious validation put the estimated elevations against our
RTK transect and found a regression slope well below one — high ground
systematically pulled down. We chased physical explanations for weeks,
each with its own control: tide-model error, storm surge, censoring at
the tidal ceiling, morphological drift between survey and archive.
Every one failed to explain it (the phase log keeps the full list,
including one retracted analysis that turned out to be circular).

In [2]:
# slope of product elevation on surveyed elevation; 1.0 = undistorted
r = load("results/m0_baseline/result.json")
print(f"pooled dev slope vs RTK: {r['pooled_dev_slope']:.3f} "
      f"(CI {np.round(r['pooled_ci'], 3)}) on {r['n_dev']} points; "
      f"{r['n_reserved_hidden']} reserved points never opened")

pooled dev slope vs RTK: 0.728 (CI [0.665 0.786]) on 227 points; 91 reserved points never opened


The resolution was the yardstick, not the method: a GNSS point is not
a 10 m pixel median. Where a pixel contains two or more RTK points,
averaging them moves the slope toward one by the exact amount that
regression attenuation predicts: with sub-pixel sampling error
$\sigma_e$ in the *predictor* and terrain variance $\sigma_z^2$, the
expected slope is

$$\beta_k=\frac{c}{1+\sigma_e^2/(k\,\sigma_z^2)},$$

for the mean of $k$ points inside the pixel — so the $k{=}1$ vs $k{=}2$
pair identifies $\sigma_e$ and the true coefficient $c$ separately. The
$\sigma_e$ obtained this way agrees with the one measured directly from
the spread between same-pixel points, which never touches the satellite. The declared diagnostic below carries
that decomposition; it changed how every later comparison in this
notebook is read.

In [3]:
r = load("results/b1_bathymetry/result.json")
print("Villaviciosa dev-RTK declared diagnostic (both variants):")
print(r["diagnostico_dev_rtk_DECLARADO"])

Villaviciosa dev-RTK declared diagnostic (both variants):
{'uniforme': {'n': 217, 'slope_dev': 0.7720373285069617, 'rmse_dev_centrado': 0.21103239241128485}, 'operador': {'n': 217, 'slope_dev': 0.7720373285069617, 'rmse_dev_centrado': 0.21103239241128485}}


### How elevations are scored — and why never RMSE alone

Three numbers, always together. **Centred RMSE** (the residual spread
after removing the median offset — datum-free, because a binary archive
cannot see datum anyway). **Slope** of estimate on truth: 1.0 means the
relief comes out at true scale; below 1 means compression — high ground
pulled down, flats flattened. **Pearson r**: the ordering. The slope is
non-negotiable because optimising RMSE alone repeatedly manufactured
flat-looking relief with *better* RMSE — a fit can win the residual
contest by refusing to commit to extreme elevations. That trap appeared
three separate times in this project (an early per-pixel dictionary, a
threshold change, and the six-parameter pixel of notebook 06), which is
why every table in this notebook carries the triad.

## 1. Three surveyed sites: does the lag correction earn its keep?
With the yardstick understood, the external test is EMODnet survey
truth at three sites chosen to stress different regimes: the Tagus (an
interior estuary with a large lag), Aveiro (a branched lagoon where
the lag grows along the channel), and Vadehavet (an open tidal flat
where a lag is measured but should not help — the method must know
when *not* to correct).

In [4]:
# per site: slope and centred RMSE, uniform level vs corrected, same pixels
for site in ("tejo", "vadehavet", "aveiro"):
    r = load(f"results/p5_{site}/result.json")
    res = r["contra_levantamiento"]
    uni = res.get("uniforme") or res["todos"]["uniforme"]
    op = res.get("operador") or res["todos"]["operador"]
    print(f"{site:10s} tau applied {np.round(r['tau_aplicado_min'],0)}")
    print(f"           uniform  slope {uni['pendiente']:.3f} RMSE {uni['rmse_centrado']:.3f}")
    print(f"           operator slope {op['pendiente']:.3f} RMSE {op['rmse_centrado']:.3f}")

tejo       tau applied [ 0. 43. 48. 46. 45. 43.]
           uniform  slope 0.635 RMSE 0.279
           operator slope 0.685 RMSE 0.247
vadehavet  tau applied [ 0. 19. 28. 26. 27. 14.]
           uniform  slope 0.536 RMSE 0.273
           operator slope 0.502 RMSE 0.281
aveiro     tau applied [ 0. 41. 41. 45. 52. 56.]
           uniform  slope 0.455 RMSE 0.590
           operator slope 0.479 RMSE 0.580


## 2. Aveiro band by band: the improvement should follow the lag

In [5]:
# Aveiro per-band anatomy: improvement grows with the detected lag
Z = np.load("results/p5_aveiro/z_scores.npz")
zu, zo, zr, s = Z["z_uni"], Z["z_op"], Z["z_ref"], Z["s_km"]
r = load("results/p5_aveiro/result.json")
ok = np.isfinite(zu) & np.isfinite(zo) & np.isfinite(zr) & np.isfinite(s)
qs = np.nanquantile(s[ok], np.linspace(0, 1, 7)); qs[0] -= 1e-9
rows = []
for k, (a, b) in enumerate(zip(qs, qs[1:])):
    m = ok & (s > a) & (s <= b)
    eu, eo = zu[m]-zr[m], zo[m]-zr[m]
    ru = np.sqrt(np.mean((eu-np.median(eu))**2))
    ro = np.sqrt(np.mean((eo-np.median(eo))**2))
    rows.append((r["centros_km"][k], r["tau_aplicado_min"][k], ru, ro))
    print(f"band {k} (s~{rows[-1][0]:.1f} km, tau {rows[-1][1]:+.0f} min): "
          f"{ru:.3f} -> {ro:.3f}  ({1000*(ru-ro):+.0f} mm)")

band 0 (s~2.4 km, tau +0 min): 0.949 -> 0.956  (-7 mm)
band 1 (s~7.2 km, tau +41 min): 0.436 -> 0.456  (-20 mm)
band 2 (s~8.7 km, tau +41 min): 0.458 -> 0.427  (+31 mm)
band 3 (s~9.7 km, tau +45 min): 0.457 -> 0.404  (+54 mm)
band 4 (s~10.5 km, tau +52 min): 0.492 -> 0.446  (+45 mm)
band 5 (s~12.6 km, tau +56 min): 0.467 -> 0.441  (+26 mm)


The RTK transect of section 0 sits in the reach where the correction
is identity by measurement, so Villaviciosa scores the elevation
engine rather than the correction; the surveyed sites above are where
the correction itself is on trial.